In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# print("dosya indiriliyor...")
# !wget -O /content/drive/MyDrive/Colab Notebooks/endomondoHR_proper.json "https://mcauleylab.ucsd.edu/public_datasets/gdrive/fitrec/endomondoHR_proper.json"

dosya indiriliyor...
--2026-09-11 09:35:25--  http://notebooks/endomondoHR_proper.json
Resolving notebooks (notebooks)... failed: Name or service not known.
wget: unable to resolve host address ‘notebooks’
--2026-09-11 09:35:25--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/fitrec/endomondoHR_proper.json
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4929126138 (4.6G) [application/json]
Saving to: ‘/content/drive/MyDrive/Colab’

/content/drive/MyDr 100%[===================>]   4.59G  23.6MB/s    in 3m 21s  

2026-09-11 09:38:47 (23.4 MB/s) - ‘/content/drive/MyDrive/Colab’ saved [4929126138/4929126138]

FINISHED --2026-09-11 09:38:47--
Total wall clock time: 3m 22s
Downloaded: 1 files, 4.6G in 3m 21s (23.4 MB/s)


## Veri kaynağı

Bu notebook, EndomondoHR (FitRec) veri setinin ham JSON dosyasını
indirip, üzerinde çalışılabilir bir CSV'ye dönüştürmenin ilk adımı.
Dosya 4.6 GB — Google Colab'ın Drive'a bağlı depolama alanında
`/content/drive/MyDrive/ai_coach/endomondoHR_proper.json` altında
saklanıyor, böylece her oturumda tekrar indirmek gerekmiyor.

In [ ]:
# !mkdir /content/drive/MyDrive/ai_coach


In [ ]:
import json
import ast, os, pickle, glob
import pandas as pd

In [ ]:
path = '/content/drive/MyDrive/ai_coach/endomondoHR_proper.json'

## Dosyanın formatı beklenenden farklı çıktı

Bu dosya standart bir JSON dosyası (tek büyük `[...]` dizisi) değil —
her SATIRI ayrı bir antrenman kaydı (JSON Lines / NDJSON benzeri bir
format). Python'da bir metin dosyasını satır satır okumak zaten her
satırı `str` (metin) olarak verir — bu, dosyanın kendisiyle ilgili bir
sorun değil, Python'ın normal davranışı; aşağıdaki hücre bunu sadece
doğruluyor.

Asıl sorun şurada: standart `json.loads()` fonksiyonu her satırı
işlemeye çalıştığında hata fırlattı. Sebebi, satırların **gerçek JSON
değil, Python sözlük (dict) yazımıyla** kaydedilmiş olması — JSON
standardı string'leri çift tırnakla (`"key"`) ister, ama bu dosyadaki
satırlar tek tırnak kullanıyor (`'key'`), ki bu Python'da geçerli ama
JSON'da geçersiz bir sözdizimi. `json.loads`, tek tırnaklı bu satırları
"bozuk JSON" olarak görüp reddediyor.

In [ ]:
n=0
with open(path) as f:
  for line in f:
    n+=1
    print(type(line))
    if n==10:
      break

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>


## İlk deneme: elle string ayırma yetersiz kaldı

`json.loads` çalışmayınca, ilk akla gelen basit çözüm satırı tek
tırnak karakterinden (`'`) bölüp parçaları elle birleştirmeyi
denemekti (`str.split("'")`). Ama bu yaklaşım kırılgan: veri
içindeki metin alanlarında (örn. `url`, `gender`) da tek tırnak
geçebilir, iç içe liste/sözlük yapıları var, sayılar ve string'ler
karışık — string'i elle parçalayıp doğru sırayla geri toparlamak,
hataya çok açık ve sürdürülemez bir yaklaşım. Aşağıdaki hücrenin
çıktısı da bunu doğruluyor: satır, anlamlı bir yapı yerine dağınık
parça parça metinlere bölünüyor.

In [ ]:

n = 0
with open(path) as f:
    for line in f:
        n += 1
        d = line.split(sep="'")
        k=0
        for i in d:
          print(k, d[k])
          k+=1
        if n == 1:
            break


0 {
1 longitude
2 : [24.64977040886879, 24.65014273300767, 24.650910682976246, 24.650668865069747, 24.649145286530256, 24.648349760100245, 24.645312326028943, 24.6447014529258, 24.644415881484747, 24.641415160149336, 24.63826850987971, 24.636211171746254, 24.634060626849532, 24.63249195367098, 24.631360983476043, 24.629418225958943, 24.625693140551448, 24.62388290092349, 24.621391966938972, 24.62025227956474, 24.61888753809035, 24.617006219923496, 24.615299999713898, 24.614676302298903, 24.612943679094315, 24.61235510185361, 24.61154105141759, 24.610843928530812, 24.61042357608676, 24.60953777655959, 24.608315024524927, 24.606816424056888, 24.605027558282018, 24.603541865944862, 24.602169329300523, 24.600177453830838, 24.599060313776135, 24.597142953425646, 24.596376596018672, 24.594232169911265, 24.592483872547746, 24.591050064191222, 24.589767884463072, 24.58789142780006, 24.583472656086087, 24.582278402522206, 24.580545527860522, 24.579513799399137, 24.57896520383656, 24.57808376289

## Çözüm: `ast.literal_eval`

Satırlar aslında geçerli **Python** sözlük/liste yazımı (tek tırnaklı
string'ler, Python `dict`/`list` sözdizimi) — sadece geçerli JSON
değiller. `ast` (Abstract Syntax Tree) kütüphanesinin `literal_eval`
fonksiyonu tam bunun için var: bir metni, `eval()` gibi rastgele kod
çalıştırmadan (güvenlik riski taşımadan), sadece sabit Python
ifadelerini (sayı, string, liste, sözlük, tuple) ayrıştırıp gerçek
Python nesnesine çevirir. Satırlar Python dict yazımıyla kaydedildiği
için, `json.loads` yerine `ast.literal_eval` kullanmak sorunu tam
olarak çözdü — aşağıdaki hücrede her satırdan `sport` alanını başarıyla
okuyabiliyoruz.

In [ ]:
with open (path,'r') as f:
  for line in f:
    w = ast.literal_eval(line)
    print(w.get('sport'))

bike
bike
bike
bike
bike (transport)
bike (transport)
bike (transport)
bike (transport)
bike
bike
bike
bike (transport)
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike (transport)
bike
bike
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
bike (transport)
run
bike (transport)
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
bike (transport)
run
bike (transport)
run
bike (transport)
run
bike (transport)
run
run
run
run
bike
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
bike (transport)
run
bike (transport)
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
run
bike
run
bike
run
run
run
run
run
run
run
run
bike (transport)
run
run
run
run
bike (transport)
run
run
bike (transport)
run
run
run
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike
bike (transport)

KeyboardInterrupt: 

## Sadece koşu antrenmanlarını çıkar

Ham veri seti "karma" — koşu dışında yürüyüş, bisiklet, yüzme gibi
birçok farklı spor türünü bir arada içeriyor. Projemiz koşu odaklı
olduğu için (aerobik/anaerobik koşu sınıflandırması), her satırı
işlerken `sport` alanına bakıp sadece `"run"` olanları tutuyoruz,
gerisini atlıyoruz. Bu filtrelemeyi, tüm veriyi belleğe yüklemeden,
dosyayı satır satır OKURKEN yapıyoruz — aşağıdaki bölümde neden bunun
zorunlu olduğunu (bellek/RAM sınırı) göreceksiniz.

In [ ]:
def extract_runs(path, out_dir, batch=20000):
    os.makedirs(out_dir, exist_ok=True)
    state = f'{out_dir}/state.txt'
    start = int(open(state).read()) if os.path.exists(state) else 0
    n_batch = len(glob.glob(f'{out_dir}/batch_*.pkl'))
    buf = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i < start:
                continue
            try:
                w = ast.literal_eval(line)
                if w.get('sport') == 'run':
                    buf.append(w)
            except Exception:
                continue
            if (i + 1) % batch == 0:
                pickle.dump(buf, open(f'{out_dir}/batch_{n_batch}.pkl', 'wb'))
                n_batch += 1
                buf.clear()
                open(state, 'w').write(str(i + 1))
                print(f'{i+1} satır islendi')
    if buf:
        pickle.dump(buf, open(f'{out_dir}/batch_{n_batch}.pkl', 'wb'))
    open(state, 'w').write(str(i + 1))

def load_runs_df(out_dir):
    dfs = [pd.DataFrame(pickle.load(open(p, 'rb')))
           for p in sorted(glob.glob(f'{out_dir}/batch_*.pkl'))]
    return pd.concat(dfs, ignore_index=True)

## RAM sınırı: neden parça parça (batch) işliyoruz

4.6 GB'lık JSON dosyasının tamamını okuyup her satırı Python nesnesine
çevirip tek bir listede/DataFrame'de biriktirmeye çalışmak, Colab'ın
sağladığı sınırlı RAM'i doldurdu ve oturum çöküp kendiliğinden yeniden
başladı (o ana kadarki tüm ilerleme kayboldu). Çözüm: dosyayı satır
satır okuyup, belirli bir sayıda satır (`batch=20000`) biriktikçe bunu
diske ayrı bir dosya (`.pkl`) olarak kaydedip bellekten temizlemek.
Böylece bellekte HER ZAMAN sadece son batch kadar veri tutuluyor, tüm
veri seti değil.

Ayrıca bir `state.txt` dosyasıyla "en son hangi satıra kadar
işlendiği" kaydediliyor — oturum yine çökerse, baştan başlamak yerine
kaldığı satırdan devam edilebiliyor. Bu, büyük/uzun süren veri
işleme görevlerinde standart bir dayanıklılık (resilience) tekniği:
işi küçük, tekrarlanabilir parçalara bölmek ve ilerlemeyi kaydetmek.

In [ ]:
extract_runs('/content/drive/MyDrive/ai_coach/endomondoHR_proper.json',
             '/content/drive/MyDrive/ai_coach/batches')
df = load_runs_df('/content/drive/MyDrive/ai_coach/batches')
print(df.shape)

20000 satır islendi
40000 satır islendi
60000 satır islendi
80000 satır islendi
100000 satır islendi
120000 satır islendi
140000 satır islendi
160000 satır islendi
(70591, 11)


## İlk gözlem: hız verisi çoğunlukla eksik

Filtrelenmiş 70591 koşu antrenmanından yalnızca ~11500'ünde (`df.info()`
çıktısındaki `speed` sütununun "non-null" sayısına bakın) doğrudan
kayıtlı bir hız verisi var — geri kalan ~60000 satırda `speed` boş
(`NaN`). Bu, projeyi durduran bir sorun değil, çünkü hız zaten konum
verisinden (enlem/boylam, zaman damgalarıyla birlikte) türetilebilir:
ardışık iki GPS noktası arasındaki mesafeyi hesaplayıp aradaki zamana
bölerek hıza ulaşılabilir.

Ama bu türetilmiş hız ham haliyle GÜRÜLTÜLÜ olacak — GPS ölçümü kusursuz
değil, sinyal sıçramaları ve tekrarlanan zaman damgaları gibi
sorunlar üretilen hızda anlamsız sıçramalara yol açar. Bu ham hızı
temizleyip kullanılabilir hale getirme işi ayrı bir notebook'ta
(`haversine_savgol_filter.ipynb`) ele alınıyor. Bu notebook'un görevi
burada bitiyor: veriyi ham JSON'dan çalışılabilir bir CSV'ye
dönüştürmek.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70591 entries, 0 to 70590
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   longitude   70591 non-null  object
 1   altitude    70591 non-null  object
 2   latitude    70591 non-null  object
 3   sport       70591 non-null  object
 4   id          70591 non-null  int64 
 5   heart_rate  70591 non-null  object
 6   gender      70591 non-null  object
 7   url         70591 non-null  object
 8   userId      70591 non-null  int64 
 9   timestamp   70591 non-null  object
 10  speed       11532 non-null  object
dtypes: int64(2), object(9)
memory usage: 5.9+ MB


In [ ]:
df.head()


,longitude,altitude,latitude,sport,id,heart_rate,gender,url,userId,timestamp,speed
0,"[6.8854929, 6.8853678, 6.8851621, 6.8848205, 6...","[-173.8, -151.2, -161.6, -165.4, -168.6, -172....","[52.2226809, 52.222727, 52.2228258, 52.2228606...",run,321063199,"[80, 81, 94, 100, 102, 112, 108, 114, 110, 109...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1397079203, 1397079210, 1397079218, 139707922...",NaN
1,"[6.9144073, 6.9142929, 6.9141539, 6.9140268, 6...","[57.8, 57.6, 57.0, 56.4, 55.8, 55.2, 54.4, 53....","[52.2111711, 52.2112631, 52.2114064, 52.211608...",run,303565793,"[60, 62, 92, 92, 132, 150, 150, 159, 159, 161,...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1393908533, 1393908541, 1393908549, 139390855...",NaN
2,"[6.9141348, 6.9145702, 6.9151684, 6.9158377, 6...","[22.8, 26.4, 30.8, 35.6, 43.0, 48.4, 49.8, 49....","[52.2110297, 52.2106325, 52.2102453, 52.209833...",run,302666522,"[77, 93, 107, 121, 118, 120, 120, 124, 124, 12...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1393687929, 1393687948, 1393687967, 139368798...",NaN
3,"[6.8678543, 6.8678634, 6.8675429, 6.8672183, 6...","[35.4, 35.2, 34.6, 34.2, 35.0, 35.2, 34.8, 34....","[52.1936673, 52.1934354, 52.1931993, 52.192873...",run,296982347,"[75, 101, 116, 120, 124, 126, 127, 129, 126, 1...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1392480163, 1392480176, 1392480189, 139248020...",NaN
4,"[6.9143328, 6.9146396, 6.9148949, 6.9151568, 6...","[63.0, 65.2, 66.0, 66.2, 65.8, 65.8, 67.0, 67....","[52.2112195, 52.2110264, 52.2108135, 52.210601...",run,295890426,"[58, 83, 112, 115, 117, 116, 141, 121, 120, 11...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1392180426, 1392180436, 1392180446, 139218045...",NaN


## Ara sonucu kaydet

Colab oturumları zaman aşımına uğrayabilir ya da beklenmedik şekilde
kapanabilir (yukarıda RAM yüzünden bir kez zaten yaşandı) — bu
durumda bellekteki `df` kaybolur ve tüm bu işlem (indirme, ayrıştırma,
filtreleme, batch'leme) baştan tekrarlanması gerekir. Bunu önlemek
için, üzerinde çalışacağımız temiz DataFrame'i kalıcı bir CSV dosyası
olarak Drive'a kaydediyoruz — bundan sonraki notebook'lar (örn.
`haversine_savgol_filter.ipynb`), bu ağır ön-işleme adımlarını tekrar
yapmadan, doğrudan bu CSV'yi okuyarak devam edebilir.

In [ ]:
df_to_csv = df.to_csv('/content/drive/MyDrive/ai_coach/endomondoHR_proper.csv')